In [5]:
import pandas as pd
import numpy as np
import os

# Create proper folder structure
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/cleaned', exist_ok=True)
print(" Folders 'data/raw/' and 'data/cleaned/' are ready!")

np.random.seed(76)

# 1. suppliers_messy.csv
suppliers = pd.DataFrame({
    'supplier_id': ['SUP102', 'SUP205', 'SUP307', 'SUP419', 'SUP521'],
    'supplier_name': [' Bosch GmbH ', 'Denso Corporation', ' Continental AG ', 'Foxconn ', 'Magna International'],
    'country': ['Germany', 'Japan', 'Germany', 'China', 'Canada'],
    'lead_time_days': [21, 18, 25, 30, 15],
    'defect_rate': [0.008, 0.005, 0.012, 0.021, 0.009],
    'contract_cost': [12500000, 8200000, 9500000, 11000000, 14500000]
})
suppliers.to_csv('data/raw/suppliers_messy.csv', index=False)

# 2. bom_messy.csv
bom = pd.DataFrame({
    'vehicle_model': ['SUV_X1']*4 + ['SED_A2']*2,
    'component_id': ['ENG204', 'TYR552', 'CHIP778', 'BOLT-991', 'BRAKE-331', 'SENS-445'],
    'component_name': ['Engine Assembly ', 'Tire', ' Chipset Module', 'Bolt Kit ', 'Brake Module', 'Sensor Assembly'],
    'quantity_per_unit': [1, 4, 12, 450, 2, 8]
})
bom.to_csv('data/raw/bom_messy.csv', index=False)

# 3. inventory_production_messy.csv
dates = pd.date_range('2026-05-14', periods=10)
plants = ['Pune_Plant', 'Chennai_Plant'] * 5
material_ids = ['CHIP778', 'ENG204', 'TYR552', 'BOLT-991', 'BRAKE-331'] * 2
inv_prod = pd.DataFrame({
    'plant': plants,
    'material_id': material_ids[:10],
    'current_stock': np.random.randint(400, 6000, 10),
    'reorder_level': np.random.randint(800, 7000, 10),
    'production_date': dates,
    'planned_units': np.random.randint(700, 1600, 10),
    'actual_units': np.random.randint(600, 1550, 10),
    'downtime_minutes': np.random.randint(0, 350, 10)
})
inv_prod.to_csv('data/raw/inventory_production_messy.csv', index=False)

# 4. logistics_messy.csv
logistics = pd.DataFrame({
    'shipment_id': ['SHP998','SHP999','SHP1000','SHP1001'],
    'origin': ['Chennai', ' Japan ', 'Germany', 'China'],
    'destination': ['Pune_Plant', 'Chennai_Plant', 'Pune_Plant', 'Chennai_Plant'],
    'mode': ['Rail', 'Sea', 'Air', 'Road'],
    'transit_days': [4, 18, 3, 12],
    'delay_days': [2, 0, 1, 5],
    'supplier_id': ['SUP102', 'SUP205', 'SUP307', 'SUP419']
})
logistics.to_csv('data/raw/logistics_messy.csv', index=False)

# 5. dealers_demand_messy.csv (for forecasting later)
demand = pd.DataFrame({
    'date': pd.date_range('2025-01-01', periods=365, freq='D'),
    'vehicle_model': np.random.choice(['SUV_X1', 'SED_A2'], 365),
    'expected_sales': np.random.randint(200, 600, 365)
})
demand.to_csv('data/raw/dealers_demand_messy.csv', index=False)

print(" SUCCESS! All 5 messy automotive files created in data/raw/")
print("Files created:")
for file in os.listdir('data/raw'):
    print("   -", file)

 Folders 'data/raw/' and 'data/cleaned/' are ready!
 SUCCESS! All 5 messy automotive files created in data/raw/
Files created:
   - bom_messy.csv
   - dealers_demand_messy.csv
   - logistics_messy.csv
   - inventory_production_messy.csv
   - suppliers_messy.csv


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ==================== SEED FOR REPRODUCIBILITY ====================
np.random.seed(76)

# ==================== 1. Suppliers ====================
suppliers_data = {
    'supplier_id': [f'SUP{str(i).zfill(3)}' for i in range(1, 26)],
    'supplier_name': [
        'Bosch Automotive', 'Continental AG', 'Denso Corporation', 'Magna International',
        'Lear Corporation', 'Valeo', 'ZF Friedrichshafen', 'Faurecia', 'Autoliv',
        'BorgWarner', 'Delphi Technologies', 'Yazaki Corporation', 'Sumitomo Electric',
        'Hyundai Mobis', 'Aisin Seiki', 'Toyoda Gosei', 'Mahle GmbH', 'Tenneco',
        'Honeywell', '3M Automotive', 'Saint-Gobain', 'Schaeffler Group', 'NTN Corporation',
        'SKF Group', 'GKN Automotive'
    ],
    'location': ['Stuttgart, Germany', 'Hanover, Germany', 'Kariya, Japan', 'Aurora, Canada',
                 'Southfield, USA', 'Paris, France', 'Friedrichshafen, Germany', 'Nanterre, France',
                 'Stockholm, Sweden', 'Auburn Hills, USA', 'Troy, USA', 'Tokyo, Japan',
                 'Osaka, Japan', 'Seoul, South Korea', 'Kariya, Japan', 'Inabe, Japan',
                 'Stuttgart, Germany', 'Lake Forest, USA', 'Charlotte, USA', 'St. Paul, USA',
                 'Courbevoie, France', 'Herzogenaurach, Germany', 'Osaka, Japan', 'Gothenburg, Sweden', 'Redditch, UK'],
    'country': ['Germany', 'Germany', 'Japan', 'Canada', 'USA', 'France', 'Germany', 'France',
                'Sweden', 'USA', 'USA', 'Japan', 'Japan', 'South Korea', 'Japan', 'Japan',
                'Germany', 'USA', 'USA', 'USA', 'France', 'Germany', 'Japan', 'Sweden', 'UK'],
    'avg_lead_time_days': np.random.randint(3, 18, 25),
    'reliability_score': np.round(np.random.uniform(0.85, 0.99), 2),
    'on_time_delivery_rate': np.round(np.random.uniform(88, 99.5), 2)
}
suppliers = pd.DataFrame(suppliers_data)
suppliers.to_csv('data/cleaned/cleaned_suppliers.csv', index=False)
print("  cleaned_suppliers.csv created (25 suppliers)")

# ==================== 2. Parts Master ====================
parts_data = {
    'part_id': [f'P{str(i).zfill(5)}' for i in range(1, 151)],
    'part_name': [
        'Engine Control Module', 'Brake Caliper Assembly', 'Front Bumper Assembly', 'LED Headlamp',
        'Transmission Gear Set', 'Catalytic Converter', 'Airbag Module', 'Power Steering Pump',
        'Radiator Assembly', 'Alternator', 'Fuel Injector Set', 'Door Lock Actuator', 'Wiper Motor',
        'Suspension Control Arm', 'Oxygen Sensor', 'Battery Pack (Hybrid)', 'Seat Belt Retractor',
        'Climate Control Panel', 'ABS Sensor', 'Drive Shaft', 'Exhaust Manifold', 'Timing Belt',
        'Spark Plug Set', 'Oil Filter', 'Brake Pad Set'
    ] * 6,  # repeat to reach 150 rows
    'category': np.random.choice(['Engine', 'Body', 'Electronics', 'Brakes', 'Suspension', 'Transmission', 'Interior'], 150),
    'supplier_id': np.random.choice(suppliers['supplier_id'], 150),
    'unit_cost_usd': np.round(np.random.uniform(2.5, 850, 150), 2),
    'standard_lead_time_days': np.random.randint(4, 16, 150),
    'abc_classification': np.random.choice(['A', 'B', 'C'], 150, p=[0.15, 0.35, 0.50]),
    'critical_for_jit': np.random.choice([True, False], 150, p=[0.25, 0.75])
}
parts_master = pd.DataFrame(parts_data)

# Fix length mismatch (in case of repeat)
parts_master = parts_master.iloc[:150].copy()
parts_master['part_name'] = parts_master['part_name'].iloc[:150].values

parts_master.to_csv('data/cleaned/cleaned_parts_master.csv', index=False)
print("  cleaned_parts_master.csv created (150 realistic parts) ← This was the missing file!")

# ==================== 3. Inventory Levels (WMS snapshot) ====================
inventory_data = {
    'part_id': parts_master['part_id'],
    'warehouse_location': np.random.choice(['WH-A1', 'WH-B2', 'WH-C3', 'WH-D4'], 150),
    'current_stock': np.random.randint(50, 2500, 150),
    'safety_stock': np.random.randint(20, 400, 150),
    'reorder_point': np.random.randint(80, 800, 150),
    'last_updated': datetime.now().date()
}
inventory_levels = pd.DataFrame(inventory_data)
inventory_levels.to_csv('data/cleaned/cleaned_inventory_levels.csv', index=False)
print("  cleaned_inventory_levels.csv created")

# ==================== 4. Purchase Orders (ERP) ====================
po_data = {
    'po_id': [f'PO{datetime.now().strftime("%y%m")}{str(i).zfill(4)}' for i in range(1, 81)],
    'part_id': np.random.choice(parts_master['part_id'], 80),
    'supplier_id': np.random.choice(suppliers['supplier_id'], 80),
    'order_quantity': np.random.randint(100, 5000, 80),
    'order_date': [datetime.now().date() - timedelta(days=np.random.randint(1, 30)) for _ in range(80)],
    'expected_delivery_date': [datetime.now().date() + timedelta(days=np.random.randint(5, 18)) for _ in range(80)],
    'actual_delivery_date': [None] * 40 + [datetime.now().date() + timedelta(days=np.random.randint(-3, 8)) for _ in range(40)],
    'status': np.random.choice(['Open', 'In Transit', 'Delivered', 'Delayed'], 80, p=[0.3, 0.25, 0.35, 0.1]),
    'unit_price': np.round(np.random.uniform(3, 800, 80), 2)
}
purchase_orders = pd.DataFrame(po_data)
purchase_orders.to_csv('data/cleaned/cleaned_purchase_orders.csv', index=False)
print("  cleaned_purchase_orders.csv created (80 realistic POs)")

print("\n  ALL FOUR cleaned files are now ready in data/cleaned/ !")

  cleaned_suppliers.csv created (25 suppliers)
  cleaned_parts_master.csv created (150 realistic parts) ← This was the missing file!
  cleaned_inventory_levels.csv created
  cleaned_purchase_orders.csv created (80 realistic POs)

  ALL FOUR cleaned files are now ready in data/cleaned/ !


In [9]:

import pandas as pd
from sqlalchemy import create_engine

# Your macOS username is pranaysingh
# If you set a password during PostgreSQL install, replace the empty part after : 
engine = create_engine('postgresql+psycopg2://pranaysingh:@localhost:5432/supply_chain')

print(" Connected to PostgreSQL successfully!")

 Connected to PostgreSQL successfully!


In [10]:
# Pull data using the new Views we created in Day 4
df_critical = pd.read_sql("SELECT * FROM autoforge.vw_critical_jit_stock", engine)
print("=== CRITICAL JIT STOCK VIEW ===")
print(df_critical.head())

df_supplier = pd.read_sql("SELECT * FROM autoforge.vw_supplier_risk", engine)
print("\n=== SUPPLIER RISK VIEW ===")
print(df_supplier)

=== CRITICAL JIT STOCK VIEW ===
  part_id               part_name     category abc_classification  \
0  P00052  Brake Caliper Assembly  Electronics                  C   
1  P00149              Oil Filter       Engine                  B   
2  P00089  Suspension Control Arm       Engine                  C   
3  P00108     Power Steering Pump     Interior                  C   
4  P00051   Engine Control Module       Brakes                  C   

   critical_for_jit  current_stock  reorder_point  safety_stock  \
0              True            219            738            21   
1              True            200            513           214   
2              True            367            648           187   
3              True            230            263           103   
4              True            691            776            42   

   percent_of_reorder                           stock_status  
0           29.674797  STOCKOUT RISK - STOP LINE IMMEDIATELY  
1           38.986355  S

In [11]:
# Run any advanced query directly from Python
query = """
WITH recent_deliveries AS (
    SELECT s.supplier_name, COUNT(*) AS total_orders,
           SUM(CASE WHEN actual_delivery_date <= expected_delivery_date THEN 1 ELSE 0 END) AS on_time
    FROM autoforge.purchase_orders po
    JOIN autoforge.suppliers s ON po.supplier_id = s.supplier_id
    WHERE actual_delivery_date IS NOT NULL
    GROUP BY s.supplier_name
)
SELECT supplier_name, total_orders, on_time,
       ROUND((on_time::DECIMAL / total_orders)*100, 2) AS otif_percent
FROM recent_deliveries
ORDER BY otif_percent DESC;
"""
df_otif = pd.read_sql(query, engine)
print(df_otif)

          supplier_name  total_orders  on_time  otif_percent
0          Toyoda Gosei             3        3         100.0
1        GKN Automotive             2        2         100.0
2         3M Automotive             3        3         100.0
3            Mahle GmbH             2        2         100.0
4        Continental AG             1        1         100.0
5     Denso Corporation             1        1         100.0
6         Hyundai Mobis             2        2         100.0
7      Bosch Automotive             2        2         100.0
8          Saint-Gobain             2        2         100.0
9                 Valeo             4        4         100.0
10            SKF Group             1        1         100.0
11          Aisin Seiki             2        2         100.0
12            Honeywell             2        2         100.0
13     Lear Corporation             3        3         100.0
14  Delphi Technologies             1        1         100.0
15  Magna International 